In [1]:

import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from uniformer import uniformer
from torchinfo import summary   
import torch
from torch import nn, einsum
from einops import rearrange
from einops.layers.torch import Reduce

device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cuda:0


In [2]:
params={
    "image_size": 224,
    "frame_size": 25,
    "num_classes": 2,
    "dim": (64, 128, 256, 512),
    "depth": (3, 4, 8, 3),
    "batch_size": 4,
    "mhsa_types": ('l', 'l', 'g', 'g'),
    "epoch": 1000,
    "data_path": '../../data/',
    "second": '5sec',
    "class_name": '물과 비누로 손위생',
    "label_path": "../../data/label/check_list/",
    "image_channel": 3
}
params["second"]=f'{params["frame_size"]//5}sec'
params

{'image_size': 224,
 'frame_size': 25,
 'num_classes': 2,
 'dim': (64, 128, 256, 512),
 'depth': (3, 4, 8, 3),
 'batch_size': 4,
 'mhsa_types': ('l', 'l', 'g', 'g'),
 'epoch': 1000,
 'data_path': '../../data/',
 'second': '5sec',
 'class_name': '물과 비누로 손위생',
 'label_path': '../../data/label/check_list/',
 'image_channel': 3}

In [3]:
model = uniformer.MultiVideoUniformer(
    num_classes = params['num_classes'],                 # number of output classes
    dims = params['dim'],         # feature dimensions per stage (4 stages)
    depths = params['depth'],              # depth at each stage
    mhsa_types = params['mhsa_types']   # aggregation type at each stage, 'l' stands for local, 'g' stands for global
).to(device)

video_size = (params['batch_size'], params['image_channel'], params['frame_size'], params['image_size'], params['image_size']) # (batch, channels, time, height, width)
optimizer = optim.AdamW(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
summary(
    model,
    input_size=[
        (params['batch_size'], params['image_channel'], params['frame_size'], params['image_size'], params['image_size']),  # video1
        (params['batch_size'], params['image_channel'], params['frame_size'], params['image_size'], params['image_size']),  # video2
        (params['batch_size'], params['image_channel'], params['frame_size'], params['image_size'], params['image_size'])   # video3
    ],
    device=device
)

Layer (type:depth-idx)                                       Output Shape              Param #
MultiVideoUniformer                                          [4, 2]                    --
├─Uniformer: 1-1                                             [4, 128]                  --
│    └─Conv3d: 2-1                                           [4, 64, 13, 56, 56]       9,280
│    └─ModuleList: 2-2                                       --                        --
│    │    └─ModuleList: 3-1                                  --                        490,298
│    │    └─ModuleList: 3-2                                  --                        2,563,312
│    │    └─ModuleList: 3-3                                  --                        20,995,584
│    │    └─ModuleList: 3-4                                  --                        30,687,744
│    └─Sequential: 2-3                                       [4, 128]                  --
│    │    └─Reduce: 3-5                                      [4,

In [4]:
logits 

NameError: name 'logits' is not defined